# Epstein–Zin Lifecycle

A two-regime lifecycle savings model with health-dependent mortality,
out-of-pocket health costs, and Epstein–Zin preferences. The agent saves over
annual periods starting at age 60, faces a two-state health Markov chain that
both raises mortality and drains wealth while health is bad, and transitions
stochastically into a terminal `dead` regime whose value is a bequest function.

The model is an example consumer block in the spirit of Atal, Fang, Karlsson &
Ziebarth (2025, *JPE* 133(6), doi:10.1086/734781). Their baseline is
time-separable expected utility (CARA in consumption, no saving, no bequests);
their robustness specification (sec. VI.D.1) has exactly the recursive
Kreps–Porteus structure used here — a CES aggregator wrapped around a
certainty-equivalent operator — with a CARA certainty equivalent. This example
uses the power-mean (CRRA) certainty equivalent instead and adds wealth,
saving, health costs, and a bequest so that there is a savings decision to
simulate; their CARA certainty equivalent is expressible as a
`QuasiArithmeticMean` with an exponential transform pair.

[View source on GitHub](https://github.com/OpenSourceEconomics/pylcm/blob/main/src/lcm_examples/epstein_zin.py)

## The recursion and how it maps onto pylcm

The Epstein–Zin recursion separates the elasticity of intertemporal substitution
from risk aversion. With current consumption $c_t$ and continuation value
$V_{t+1}$:

```{math}
V_t = \Bigl[(1-\beta)\,c_t^{\rho} + \beta\,\mathrm{CE}_t^{\rho}\Bigr]^{1/\rho},
\qquad
\mathrm{CE}_t = \Bigl(\mathbb{E}_t\bigl[V_{t+1}^{\,1-\gamma}\bigr]\Bigr)^{1/(1-\gamma)}
```

where $\beta$ is the discount factor, $\rho = 1 - 1/\psi$ with $\psi$ the
elasticity of intertemporal substitution, and $\gamma$ is the coefficient of
relative risk aversion. When $\gamma = \rho$ the recursion collapses to expected
CRRA utility.

The mapping onto pylcm has four parts:

- **`utility`** returns $c_t$ directly. The aggregator below is a weighted power
  *mean* of `utility` and the certainty equivalent, so the value function lives
  in (positive) consumption units — per-period utility must too.
- **`H`** is the outer Kreps–Porteus aggregator. Pass the built-in
  `lcm.H_epstein_zin` via `functions={"H": H_epstein_zin, ...}`; it computes
  `((1 − β) · utility^ρ + β · E_next_V^ρ)^(1/ρ)`, where the `E_next_V` argument
  receives the certainty equivalent already computed by the engine. It is
  parametrized directly by the IES:
  `params["alive"]["H"]["intertemporal_elasticity_of_substitution"]` ($\psi$,
  with curvature $\rho = 1 - 1/\psi$ computed inside). The expected-utility
  counterpart `lcm.H_linear` (`utility + β · E_next_V`) is the default when no
  `H` is supplied.
- **`certainty_equivalent=PowerMean()`** on the `Regime` implements
  $\mathrm{CE}_t$ with the power transform $g(v) = v^{1-\gamma}$. Its runtime
  parameter lives at
  `params["alive"]["certainty_equivalent"]["risk_aversion"]`.
- The **expectation** runs jointly over the health Markov chain and the alive/dead
  regime transition:
  $\mathrm{CE} = g^{-1}\!\bigl(\sum_r p_r\,\mathbb{E}_w[g(V'_r)]\bigr)$. The
  engine handles the two-level expectation automatically given the stochastic
  regime transition and the stochastic health transition in
  `state_transitions`.

## Pitfalls

- **Positivity.** Power transforms require $V' > 0$ everywhere: with $\gamma > 1$,
  $g(0) = \infty$ propagates backward and corrupts the entire solution. Implicit
  zeros count too — a reachable target regime with no states contributes $0$ to
  the transformed sum, which is coherent only when $g(0) = 0$. Here every value
  is positive by construction: utility is consumption itself (bounded below by
  the consumption-grid floor), the bequest is strictly positive at every
  reachable wealth, and `next_wealth` clips to the wealth grid.
- **Scale the bequest.** Because `H_epstein_zin` is a weighted power *mean*, the
  alive value sits at the scale of *per-period* consumption. A terminal value in
  wealth-like units — an unscaled `sqrt(wealth)`, say — easily exceeds it, which
  makes death the *good* branch of the certainty equivalent, and a more
  risk-averse agent then runs its wealth *down* toward the certain bequest.
  Price the bequest in consumption-equivalent units (here
  `bequest_scale * sqrt(wealth)` with `bequest_scale = 0.3`) so death stays the
  bad branch.
- **`risk_aversion = 1`** is the geometric-mean (log) limit of the power mean,
  $\exp(\mathbb{E}[\log V'])$; `PowerMean` handles it directly.
- **Solver restriction.** `certainty_equivalent` is supported only with the
  `GridSearch` solver. Passing `certainty_equivalent` together with a DC-EGM solver
  is rejected at model build.

## Run it

Build a 20-period model on fine wealth and consumption grids, then solve and
simulate 1,000 subjects for two Epstein–Zin parametrizations that share the
intertemporal elasticity of substitution ($\psi = 2$) and differ only in risk
aversion ($\gamma = 0.5$ versus $\gamma = 5$). Income (1.0) exceeds the
consumption floor (0.5), so agents can save; while health is bad they pay an
out-of-pocket cost of 1.5 per period, and bad health is persistent — an
uninsurable expense risk in the spirit of the paper's medical spending.

In [ ]:
import jax.numpy as jnp
import numpy as np
import plotly.graph_objects as go

from lcm import PowerMean
from lcm_examples.epstein_zin import EZRegimeId, HealthStatus, get_model, get_params

n_periods = 20
survival_probs = tuple(np.concatenate([np.linspace(0.99, 0.85, n_periods - 2), [0.0]]))
model = get_model(
    certainty_equivalent=PowerMean(),
    n_periods=n_periods,
    n_wealth_points=60,
    n_consumption_points=60,
)

In [ ]:
n_subjects = 1_000
labels = {
    0.5: "risk tolerant: γ = 0.5, ψ = 2",
    5.0: "risk averse: γ = 5, ψ = 2",
}
mean_wealth_by_age = {}
for risk_aversion in labels:
    params = get_params(
        risk_aversion=risk_aversion,
        discount_factor=0.95,
        intertemporal_elasticity_of_substitution=2.0,
        survival_probs=survival_probs,
        income=1.0,
        health_cost=1.5,
        bequest_scale=0.3,
    )
    result = model.simulate(
        params=params,
        initial_conditions={
            "age": jnp.full(n_subjects, 60.0),
            "wealth": jnp.tile(jnp.linspace(1.0, 10.0, 100), n_subjects // 100),
            "health": jnp.full(n_subjects, HealthStatus.good, dtype=jnp.int32),
            "regime_id": jnp.full(n_subjects, EZRegimeId.alive, dtype=jnp.int32),
        },
        period_to_regime_to_V_arr=None,
        log_level="off",
        seed=42,
    )
    df = result.to_dataframe()
    alive = df[df["regime_name"] == "alive"]
    mean_wealth_by_age[risk_aversion] = alive.groupby("age")["wealth"].mean()

In [ ]:
colors = {0.5: "#999999", 5.0: "#e63946"}

fig = go.Figure()
for risk_aversion, series in mean_wealth_by_age.items():
    fig.add_trace(
        go.Scatter(
            x=series.index,
            y=series.to_numpy(),
            mode="lines",
            name=labels[risk_aversion],
            line={"color": colors[risk_aversion], "width": 2.5},
            showlegend=False,
        )
    )
    fig.add_annotation(
        x=series.index[-1],
        y=series.to_numpy()[-1],
        text=labels[risk_aversion],
        showarrow=False,
        xanchor="left",
        xshift=6,
        font={"color": colors[risk_aversion]},
    )
fig.update_layout(
    title="Mean wealth among survivors by age",
    xaxis_title="Age",
    yaxis_title="Mean wealth",
    template="simple_white",
    margin={"r": 220},
)
fig.show()

Both agents run down their initial wealth early in retirement: wealth earns no
return, so with $\beta \cdot p_{\text{survive}} < 1$ impatience dominates while
buffers are ample. The difference emerges as the buffers approach their target
levels. The risk-averse agent ($\gamma = 5$) holds — and from the late 60s on
rebuilds — a visibly larger wealth buffer: a persistent bad-health spell drains
1.5 per period against an income of 1.0, and dying is worst at low wealth
(the bequest is concave), so wealth is precautionary insurance against both.
The risk-tolerant agent ($\gamma = 0.5$) is content with a smaller buffer and a
lower late-life wealth path. Only risk aversion differs between the two
parametrizations — the IES, and with it the willingness to shift consumption
across time, is identical — so the entire gap is a precautionary response to
health-cost and mortality risk.

Two features of this comparison are worth flagging. First, the precautionary
ranking is *not* automatic in models like this one: it holds because wealth can
effectively self-insure the bad-health spell. Make the spell expensive enough
relative to income and feasible buffers (say, an out-of-pocket cost above 2.5
here) and the ranking flips — the risk-averse agent gives up on buffering the
now-uninsurable state and consumes its wealth instead. Second, the mean is
taken over *survivors*, and survival is health-dependent, so late-life ages
average over a healthier-than-average, and mechanically wealthier, surviving
population.